# 12_04 — Evaluación predictiva · Escenario 2

Paso **3b** del ciclo. Supone que `12_03_convergencia` dio veredicto ✓; si las
cadenas no convergieron, nada de lo que sigue significa algo.

## Qué se reporta

| Sección | Qué responde |
|---|---|
| 4 | métricas puntuales y distribucionales, **train vs test** |
| 5 | bandas de credibilidad sobre la serie de cada score |
| 6 | intervalos sobre la curva, en extractos cada 10 períodos |
| 7 | ventana móvil: cómo evoluciona el error al cruzar $T_0$ |
| 8 | calibración marginal: PIT por bloque |
| **9** | **calibración condicional al nivel de volatilidad verdadero** |
| 10 | comparación con las líneas base |

Todo el cálculo vive en `fit/` y todo el dibujo en `graphics/`: este notebook
sólo orquesta.

## Cómo leer este escenario

En el Algoritmo 2 la media condicional es constante, $\mathbb{E}[X_t\mid
X_{t-1}]=\mu$. De ahí se siguen tres advertencias de lectura que conviene
tener presentes antes de mirar cualquier número:

1. **El RMSE y el $R^2$ no discriminan.** La predicción puntual óptima es la
   media incondicional; un $R^2$ cercano a cero es el resultado *correcto*, no
   un fracaso. Léase como control: un $R^2$ marcadamente negativo sí indicaría
   que el modelo está inventando estructura en la media.
2. **La comparación importa en la forma de la predictiva**: CRPS, puntaje de
   energía y ancho de banda. Ahí una predictiva de ancho constante —la que
   entrega cualquier método homocedástico— pierde frente a una que se estrecha
   en calma y se ensancha en estrés.
3. **La cobertura marginal puede engañar.** Bandas demasiado anchas en calma y
   demasiado angostas en estrés promedian al 95 % nominal. Por eso §9
   estratifica la cobertura según el nivel de volatilidad **verdadero** del
   generador, que es el eje 2 del diseño (`docs/03 Modelo.tex §03_06`) y el
   resultado central de esta corrida.

## 1. Imports, rutas y artefactos

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat

from model_psbp_fd.pipelines import (
    cargar_curvas, cargar_curvas_true, cargar_fpca, cargar_estandarizador,
    cargar_datasets_ar, cargar_hiperparametros, cargar_config_evaluacion,
    cargar_escenario,
)
from model_psbp_fd.functions_models import DataStandardizer
from model_psbp_fd.models.pspb_fd_v3 import PSBPPredictor, PropagadorFuncional

from model_psbp_fd.fit import (
    # puntual
    rmse_por_coeficiente, r2_por_columna, razon_dispersion, mise, rmse_funcional,
    # distribucional muestral
    crps_muestral, energy_score, cobertura, intervalo_muestral,
    pit_muestral, diagnostico_pit,
    # calibración condicional al estado verdadero (eje 2 del diseño)
    estratos_por_cuantil, cobertura_condicional,
    # agrupación de cadenas y ventana móvil
    agrupar_momentos, ventana_movil_scores, ventana_movil_funcional,
)
from model_psbp_fd.graphics import (
    plot_ventana_movil, plot_bandas_serie, plot_extractos_curvas,
    plot_calibracion_pit, plot_scatter_theta,
)
from model_psbp_fd.utils import get_project_root

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

In [ ]:
PROJECT_ROOT = get_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# [CONFIG] debe coincidir con 12_01, 12_03 y psbp_fd_iteracion.m
BASENAME, ESCENARIO_ID, REPLICA_ID = "escenario", 2, 1
EXPERIMENT_ID = f"{BASENAME}_{ESCENARIO_ID}_r{REPLICA_ID:02d}"

PATHS = {
    "raw":          PROJECT_ROOT / "data" / "simulaciones" / "raw" / EXPERIMENT_ID,
    "functional":   PROJECT_ROOT / "data" / "simulaciones" / "processed" / "functional" / EXPERIMENT_ID,
    "predict":      PROJECT_ROOT / "data" / "simulaciones" / "processed" / "predict" / EXPERIMENT_ID,
    "out_report":   PROJECT_ROOT / "reports" / "simulaciones" / EXPERIMENT_ID,
    "out_artefact": PROJECT_ROOT / "artefact" / "simulaciones" / EXPERIMENT_ID,
}
for r in PATHS.values():
    r.mkdir(parents=True, exist_ok=True)
print(f"EXPERIMENT_ID : {EXPERIMENT_ID}")

In [ ]:
dfs_train, manifest = cargar_datasets_ar(PATHS, bloque="train")
dfs_test,  _        = cargar_datasets_ar(PATHS, bloque="test")
hp_json     = cargar_hiperparametros(PATHS)
eval_config = cargar_config_evaluacion(PATHS)

COMPONENT_IDX = manifest["component_idx"]
n_components  = len(COMPONENT_IDX)
cov_names     = manifest["cov_names"]
N_LAGS        = int(manifest["n_lags"])
T, T0         = int(manifest["T"]), int(manifest["T0"])
N_ITER        = int(hp_json["n_iter"])
MCMC_CFG      = hp_json["mcmc_config"]
BURN          = int(MCMC_CFG["burn"])

assert manifest["scores_scale"] == "standardized_zscore_ddof0"

# Parámetros de evaluación: se LEEN del artefacto, no se redeclaran aquí.
NIVEL        = float(eval_config.get("nivel_credibilidad", 0.95))
MODO_RESIDUO = eval_config.get("modo_residuo", "ninguno")
OBJETIVO     = eval_config.get("objetivo_evaluacion", "curva_verdadera")
VENTANAS_W   = list(eval_config.get("ventana_movil", {}).get("w", [10, 20, 40]))
ESTRAT_CFG   = eval_config.get("estratificacion", {})

print(f"T={T}  T0={T0}  n_lags={N_LAGS}  componentes={n_components}")
print(f"nivel={NIVEL}  ·  modo_residuo={MODO_RESIDUO!r}  ·  objetivo={OBJETIVO!r}")
print(f"ventanas w = {VENTANAS_W}")
print(f"estratificación = {ESTRAT_CFG.get('variable', '—')} "
      f"en {ESTRAT_CFG.get('n_estratos', '—')} estratos")

In [ ]:
fpca = cargar_fpca(PATHS)
_ver = fpca.verificar()
assert _ver["todo_ok"], f"Las identidades FPCA no se cumplen: {_ver}"

scores_standardizer = cargar_estandarizador(PATHS, DataStandardizer)
X_obs, grilla = cargar_curvas(PATHS)                 # (T, G) con ruido
X_true        = cargar_curvas_true(PATHS)            # (T, G) verdadera — el objetivo

M_fpca   = fpca.M
Psi_grid, mu_grid = fpca.Psi_grid, fpca.mu_grid
SCORES     = fpca.SCORES                              # (T, M) escala original
SCORES_STD = scores_standardizer.transform(SCORES)

print(f"FPCA M={M_fpca} K={fpca.K}   ·   curvas {X_true.shape}  grilla {grilla.shape}")
print(f"sd(observada − verdadera) = {(X_obs - X_true).std():.4f}   "
      "← el ruido que el modelo NO debe predecir")
assert COMPONENT_IDX == list(range(M_fpca)), (
    "La propagación funcional necesita el vector completo de scores en orden; "
    f"COMPONENT_IDX={COMPONENT_IDX} y M={M_fpca}.")

### 1.1 Estado verdadero del generador

La varianza condicional latente $\sigma_t^2$ es lo que hace evaluable el eje 2
en este escenario. Se lee del CSV que escribió `12_01`; si faltara, se
reconstruye del `.npz` crudo, que la conserva bajo `interno_sigma2` gracias a
`incluir_internos=True`.

Esta cantidad **no** entra en ninguna predicción: sólo define la partición de
los orígenes en §9.

In [ ]:
_csv_estado = PATHS["out_report"] / "10_estado_volatilidad.csv"
if _csv_estado.exists():
    estado_df  = pd.read_csv(_csv_estado)
    sigma2_med = estado_df["sigma2_media"].to_numpy()
    _fuente    = _csv_estado.name
else:
    _crudo = cargar_escenario(str(PATHS["raw"] / f"escenario_{ESCENARIO_ID}.npz"))
    assert "interno_sigma2" in _crudo, (
        "El .npz no contiene `interno_sigma2`: regenera 12_01 con "
        "guardar_escenario(..., incluir_internos=True).")
    sigma2_med = _crudo["interno_sigma2"][REPLICA_ID - 1].mean(axis=1)
    _fuente    = f"escenario_{ESCENARIO_ID}.npz::interno_sigma2"

assert sigma2_med.shape == (T,), f"sigma2_media tiene {sigma2_med.shape}, se esperaba ({T},)"
print(f"estado verdadero leído de {_fuente}   ·   {sigma2_med.shape}")
print(f"sigma^2 medio: [{sigma2_med.min():.3f}, {sigma2_med.max():.3f}]   "
      f"cv temporal = {sigma2_med.std()/sigma2_med.mean():.3f}")

## 2. Trazas MCMC

In [ ]:
def ruta_traza(fpc_idx, chain):
    return PATHS["out_artefact"] / f"chain_fpc_{fpc_idx}_iter{chain:02d}.mat"


def leer_traza(path):
    m = loadmat(str(path))
    claves = ["betajhout", "beta0hout", "tauhout", "alphahout", "psijhout",
              "Gammajhout", "gammajhout", "pijout", "wjout", "osumout", "inEout"]
    traces = {k: np.asarray(m[k], dtype=np.float64) for k in claves}
    burn = int(np.asarray(m["burn"]).ravel()[0])
    feat = str(np.atleast_1d(m["feature_names"]).ravel()[0]).split(",")
    return traces, burn, feat


class ModeloTraza:
    def __init__(self, traces, burn, feature_names):
        self.traces = traces
        self.feature_names_ = list(feature_names)
        self.burn = int(burn)
        self.predictor_ = PSBPPredictor(traces=traces, burn=burn)

    def _diseno(self, df):
        Xp = np.asarray(df.iloc[:, 1:], dtype=float)
        return np.hstack([np.ones((Xp.shape[0], 1)), Xp])

    def momentos(self, df):
        return self.predictor_.momentos_predictivos(self._diseno(df))

    def muestrear(self, df, d=1, seed=None):
        return self.predictor_.muestrear(self._diseno(df), d, seed=seed)


faltan = [ruta_traza(COMPONENT_IDX[k] + 1, c + 1).name
          for k in range(n_components) for c in range(N_ITER)
          if not ruta_traza(COMPONENT_IDX[k] + 1, c + 1).exists()]
assert not faltan, f"Faltan trazas: {faltan}. Ejecuta psbp_fd_iteracion.m."

models_chains = {k: {} for k in range(n_components)}
for k in range(n_components):
    esperado = list(dfs_train[k].columns[1:])
    for c in range(N_ITER):
        traces, burn, feat = leer_traza(ruta_traza(COMPONENT_IDX[k] + 1, c + 1))
        assert feat == esperado, f"[k={k} c={c+1}] feature_names ≠ columnas del dataset."
        models_chains[k][c] = ModeloTraza(traces, burn, feat)

n_post = MCMC_CFG["nsim"] - BURN
print(f"✓ {n_components} × {N_ITER} cadenas · {n_post} draws posteriores c/u")

## 3. Predicción a $h=1$ sobre la serie completa

Los dos bloques se concatenan en **una sola serie de orígenes**
$t=N_{lags}+1,\dots,T$. Sin el tramo de entrenamiento no hay con qué comparar
el salto en $T_0$, que es la lectura principal de la ventana móvil.

En todos los orígenes la predicción usa los **rezagos reales** —nunca
predicciones encadenadas—, de modo que el horizonte es 1 en todo el recorrido y
el modelo no se reentrena en ningún punto.

In [ ]:
# Número de extracciones de la predictiva por draw posterior. El total es
#     S = (nsim - burn) × S_POR_ITER × n_cadenas
# y d=1 ya da varios miles: subirlo sólo reduce el error Monte Carlo de estimar
# los cuantiles, no cambia la predictiva.
S_POR_ITER = 100
SEED_PRED  = 20260823

# Serie completa de orígenes: train (t = N_LAGS+1 … T0) seguido de test.
dfs_full = {k: pd.concat([dfs_train[k], dfs_test[k]], ignore_index=True)
            for k in range(n_components)}
n_orig   = len(dfs_full[0])
t_orig   = np.arange(N_LAGS + 1, T + 1)          # tiempo del experimento, base-1
assert len(t_orig) == n_orig, f"{len(t_orig)} ≠ {n_orig}"

# Índice del corte dentro de la serie de orígenes (los primeros N_LAGS no existen)
T0_orig  = T0 - N_LAGS
es_train = t_orig <= T0

# Estado verdadero alineado a los orígenes evaluados
sigma2_ev = sigma2_med[N_LAGS:]
assert sigma2_ev.shape[0] == n_orig

print(f"orígenes evaluados: {n_orig}  (train {es_train.sum()} · test {(~es_train).sum()})")

In [ ]:
# Momentos y extracciones por score, agrupando cadenas.
#   momentos : ley de varianza total entre cadenas (fit.agrupar_momentos)
#   muestras : concatenación, que es la mezcla de igual peso
Y_obs  = np.column_stack([dfs_full[k].iloc[:, 0].to_numpy() for k in range(n_components)])
Y_hat  = np.empty_like(Y_obs)
Y_sd   = np.empty_like(Y_obs)
draws  = []          # por componente: (S, n_orig)

for k in range(n_components):
    medias, sds, muestras_k = [], [], []
    for c in sorted(models_chains[k]):
        mom = models_chains[k][c].momentos(dfs_full[k])
        medias.append(mom["media"])
        sds.append(mom["sd"])            # PREDICTIVA (v3), no la del centro
        muestras_k.append(models_chains[k][c].muestrear(
            dfs_full[k], S_POR_ITER, seed=SEED_PRED + 1000 * k + c))
    Y_hat[:, k], Y_sd[:, k] = agrupar_momentos(np.column_stack(medias),
                                               np.column_stack(sds))
    draws.append(np.concatenate(muestras_k, axis=0))

SC_draws = np.stack(draws, axis=2)                  # (S, n_orig, M)
S_total  = SC_draws.shape[0]
li_s, ls_s = intervalo_muestral(SC_draws, nivel=NIVEL)   # (n_orig, M)

print(f"extracciones por score: S = {S_total} "
      f"= {n_post} draws × {S_POR_ITER} × {N_ITER} cadenas")
print(f"SC_draws {SC_draws.shape}  ·  bandas {li_s.shape}")

# Señal temprana del rasgo que este escenario mide: si el modelo capta la
# heterocedasticidad, el ancho de sus bandas debe seguir a sigma_t verdadera.
_ancho_s = (ls_s - li_s).mean(axis=1)
print(f"\ncorr(ancho de banda, sigma_t verdadera) = "
      f"{np.corrcoef(_ancho_s, np.sqrt(sigma2_ev))[0,1]:+.3f}")
print("   ← positiva y apreciable ⇒ la predictiva se adapta a la volatilidad;")
print("     ≈ 0 ⇒ el modelo entrega una banda de ancho esencialmente constante.")

### 3.1 Propagación a la curva

$\hat X^{(s)}_t(\tau)=\mu(\tau)+\sum_m (d_m\tilde\xi^{(s)}_{tm}+c_m)\psi_m(\tau)$,
que es (3.41) del documento. Con `modo_residuo="ninguno"` el mapa es
determinista y las curvas son extracciones exactas de la predictiva de la curva
**proyectada**.

Las muestras funcionales se **adelgazan** antes de propagar: el arreglo completo
sería $(S, n, G)$ y con $S$ de varios miles ocupa más de un gigabyte sin que los
cuantiles al 95 % mejoren de forma apreciable.

In [ ]:
# Adelgazamiento para la propagación funcional. Con S_FUNC ≈ 500 el error
# Monte Carlo de un cuantil al 2.5% es del orden de 0.01 en escala de cuantil,
# despreciable frente al ancho de la banda.
S_FUNC = 500
paso_thin = max(1, S_total // S_FUNC)
SC_thin = SC_draws[::paso_thin]

propagador = PropagadorFuncional(Psi_grid, mu_grid,
                                 estandarizador=scores_standardizer,
                                 modo_residuo=MODO_RESIDUO)
X_draws = propagador.curvas_desde_scores(SC_thin, seed=SEED_PRED)   # (S', n, G)
X_pred  = X_draws.mean(axis=0)                                       # (n, G)
li_f, ls_f = np.quantile(X_draws, (1 - NIVEL) / 2, axis=0), \
             np.quantile(X_draws, 1 - (1 - NIVEL) / 2, axis=0)

# Curvas de referencia alineadas a los orígenes evaluados
X_true_ev = X_true[N_LAGS:]     # (n_orig, G) VERDADERA — contra esto se evalúa
X_obs_ev  = X_obs[N_LAGS:]      # (n_orig, G) observada, sólo para las figuras

print(f"muestras funcionales {X_draws.shape}  (adelgazado 1 de cada {paso_thin})")
print(f"memoria ≈ {X_draws.nbytes / 1e6:.0f} MB")

# Piso de error: la mejor curva alcanzable con M componentes, sin modelo alguno.
X_proj = fpca.reconstruct(SCORES)[N_LAGS:]
print(f"\nMISE del TRUNCAMIENTO (proyección FPCA de la curva verdadera): "
      f"{mise(X_true_ev, X_proj, grilla):.6f}")
print("   ← ningún modelo sobre esta representación puede bajar de ahí")

## 4. Métricas, entrenamiento contra prueba

Recuérdese la advertencia de la cabecera: **RMSE y $R^2$ no discriminan aquí**.
La media condicional es constante, de modo que $R^2\approx 0$ es el resultado
correcto y el `sd_ratio` —razón entre la dispersión predicha y la observada— es
más informativo: un valor cercano a uno indica que la predictiva reproduce la
escala del proceso, y las desviaciones señalan bandas sistemáticamente anchas
o angostas. Las cifras que sí separan modelos son CRPS y cobertura.

In [ ]:
def _metricas_bloque(mask, etiqueta):
    """Métricas puntuales y distribucionales sobre un subconjunto de orígenes."""
    filas = []
    for k in range(n_components):
        y, p = Y_obs[mask, k], Y_hat[mask, k]
        z    = SC_draws[:, mask, k]
        cob  = cobertura(y, li_s[mask, k], ls_s[mask, k])
        pit  = pit_muestral(y, z)
        filas.append({
            "bloque":  etiqueta,
            "FPC":     f"FPC {COMPONENT_IDX[k] + 1}",
            "RMSE":    float(rmse_por_coeficiente(y[:, None], p[:, None])[0]),
            "R2":      float(r2_por_columna(y[:, None], p[:, None], centrar=True)[0]),
            "sd_ratio": float(razon_dispersion(y[:, None], p[:, None])[0]),
            "CRPS":    float(crps_muestral(y, z).mean()),
            f"Cob{int(NIVEL*100)}": cob["cobertura"],
            "Ancho":   cob["ancho_medio"],
            "PIT_ks":  float(diagnostico_pit(pit)["ks"]),
            "PIT_forma": diagnostico_pit(pit)["forma"],
            "n":       int(mask.sum()),
        })
    return filas

met_df = pd.DataFrame(_metricas_bloque(es_train, "train")
                      + _metricas_bloque(~es_train, "test"))
met_df = met_df.set_index(["bloque", "FPC"])
met_df.to_csv(PATHS["out_report"] / "50_metricas_scores.csv")

_num = [c for c in met_df.columns if met_df[c].dtype.kind == "f"]
display(met_df.style
    .format({c: "{:.4f}" for c in _num})
    .background_gradient(subset=["RMSE", "CRPS"], cmap="RdYlGn_r")
    .background_gradient(subset=[f"Cob{int(NIVEL*100)}"], cmap="RdYlGn",
                         vmin=0.80, vmax=1.0)
    .set_caption("Métricas por score y bloque · CRPS y cobertura desde la "
                 "predictiva MUESTRAL"))

_deg = (met_df.loc["test", "RMSE"].mean() / met_df.loc["train", "RMSE"].mean())
print(f"\nRMSE test / RMSE train = {_deg:.3f}")
print("   Interpretar con cuidado: en este escenario la razón mezcla el efecto "
      "de salir\n   de muestra con el del nivel de volatilidad de cada bloque "
      "(ver §2.6 de 12_01).")
print(f"R2 medio (test) = {met_df.loc['test', 'R2'].mean():+.4f}   "
      "← se espera ≈ 0; muy negativo indicaría estructura inventada en la media")

In [ ]:
# Métricas funcionales, contra la curva VERDADERA
filas_f = []
for mask, etq in ((es_train, "train"), (~es_train, "test")):
    dentro = (X_true_ev[mask] >= li_f[mask]) & (X_true_ev[mask] <= ls_f[mask])
    filas_f.append({
        "bloque": etq,
        "MISE":   mise(X_true_ev[mask], X_pred[mask], grilla),
        "RMSE_f": rmse_funcional(X_true_ev[mask], X_pred[mask], grilla),
        "MISE_truncamiento": mise(X_true_ev[mask], X_proj[mask], grilla),
        "energy": energy_score(X_true_ev[mask], X_draws[:, mask, :],
                               max_pares=2000, seed=0),
        f"Cob{int(NIVEL*100)}_puntual": float(dentro.mean()),
        "ancho_medio": float((ls_f[mask] - li_f[mask]).mean()),
        "n": int(mask.sum()),
    })
fun_df = pd.DataFrame(filas_f).set_index("bloque")
fun_df.to_csv(PATHS["out_report"] / "51_metricas_funcionales.csv")

display(fun_df.style.format("{:.6f}", subset=["MISE", "RMSE_f", "MISE_truncamiento"])
        .format("{:.4f}", subset=["energy", f"Cob{int(NIVEL*100)}_puntual", "ancho_medio"])
        .set_caption("Métricas funcionales contra la curva VERDADERA · "
                     "banda sin residuo de representación"))

_frac = fun_df["MISE_truncamiento"] / fun_df["MISE"]
print("\nfracción del MISE atribuible al truncamiento FPCA:")
for b in fun_df.index:
    print(f"  {b:5s}: {_frac[b]:.1%}"
          + ("   ← el techo de la representación domina; subir M rinde más "
             "que mejorar el modelo" if _frac[b] > 0.7 else ""))
print(f"\nLa cobertura es PUNTUAL (cada tau por separado), NO simultánea sobre "
      f"la curva.")

## 5. Bandas de credibilidad sobre la serie de scores

In [ ]:
plot_bandas_serie(
    Y_obs, Y_hat, li_s, ls_s, T0, t=t_orig,
    etiquetas=[f"FPC {i + 1}" for i in COMPONENT_IDX], nivel=NIVEL,
    title="Bandas de credibilidad por score — entrenamiento y prueba",
    save_path=str(PATHS["out_report"] / "52_bandas_scores.png"))
plt.show()
print("En este escenario la lectura de esta figura es el ANCHO de la banda, no "
      "su centro:\nel centro debe ser plano y la banda debe respirar con la "
      "volatilidad del proceso.")

In [ ]:
eval_scatter = {k: {"y_obs": Y_obs[~es_train, k], "y_hat": Y_hat[~es_train, k]}
                for k in range(n_components)}
plot_scatter_theta(eval_scatter, n_components=n_components,
                   save_path=str(PATHS["out_report"] / "53_scatter_scores_test.png"))
plt.show()
print("Nube sin pendiente = media condicional constante correctamente estimada. "
      "Es el\nresultado esperado del Algoritmo 2, no un defecto del ajuste.")

## 6. Intervalos de credibilidad sobre la curva

Extractos de la serie funcional cada `CADA` períodos, cada uno con su banda
puntual, la curva verdadera y —en gris— los datos observados, para ver de un
vistazo dónde estaba el ruido que la banda **no** tiene que cubrir.

In [ ]:
CADA = 10   # [CONFIG] un extracto cada CADA períodos

plot_extractos_curvas(
    X_true_ev, X_pred, li_f, ls_f, grilla, T0, t=t_orig,
    cada=CADA, n_col=5, nivel=NIVEL, X_obs=X_obs_ev,
    title="Predictiva funcional y banda de credibilidad",
    save_path=str(PATHS["out_report"] / "54_extractos_curvas.png"))
plt.show()

In [ ]:
# Zoom sobre la frontera: los últimos orígenes de train y los primeros de test.
i_corte = int(np.searchsorted(t_orig, T0))
sel = np.arange(max(0, i_corte - 3), min(n_orig, i_corte + 4))

plot_extractos_curvas(
    X_true_ev[sel], X_pred[sel], li_f[sel], ls_f[sel], grilla, T0,
    t=t_orig[sel], cada=1, n_col=len(sel), nivel=NIVEL, X_obs=X_obs_ev[sel],
    title=f"Frontera train/test (T0={T0})",
    save_path=str(PATHS["out_report"] / "55_extractos_frontera.png"))
plt.show()

In [ ]:
# Extractos elegidos por ESTADO y no por calendario: los tres orígenes de
# prueba con menor volatilidad verdadera y los tres con mayor. Es la lectura
# visual del resultado de §9.
idx_test = np.where(~es_train)[0]
orden    = idx_test[np.argsort(sigma2_ev[idx_test])]
sel_vol  = np.concatenate([orden[:3], orden[-3:]])

plot_extractos_curvas(
    X_true_ev[sel_vol], X_pred[sel_vol], li_f[sel_vol], ls_f[sel_vol], grilla, T0,
    t=t_orig[sel_vol], cada=1, n_col=6, nivel=NIVEL, X_obs=X_obs_ev[sel_vol],
    title="Prueba: 3 orígenes de menor volatilidad (izq.) y 3 de mayor (der.)",
    save_path=str(PATHS["out_report"] / "56_extractos_por_volatilidad.png"))
plt.show()

_a_baja = (ls_f[orden[:3]] - li_f[orden[:3]]).mean()
_a_alta = (ls_f[orden[-3:]] - li_f[orden[-3:]]).mean()
print(f"ancho medio de la banda: baja volatilidad {_a_baja:.3f}  ·  "
      f"alta {_a_alta:.3f}  ·  razón {_a_alta/_a_baja:.2f}×")
print(f"razón de sigma verdadera entre esos mismos orígenes: "
      f"{np.sqrt(sigma2_ev[orden[-3:]].mean()/sigma2_ev[orden[:3]].mean()):.2f}×")
print("   ← si ambas razones se parecen, el modelo está siguiendo la "
      "volatilidad verdadera.")

## 7. Ventana móvil

El modelo **no se reentrena**: lo que se desliza es la ventana de evaluación.
Cada punto agrega las métricas de $w$ orígenes consecutivos, todos a $h=1$ con
los rezagos reales. Se superponen varios anchos para que la conclusión no
dependa de un $w$ elegido a dedo; las ventanas que **cruzan** $T_0$ van
punteadas, porque su cifra mezcla dentro y fuera de muestra.

En este escenario la ventana móvil tiene una lectura adicional: el MISE local
debería subir y bajar **siguiendo a $\sigma_t^2$**, no de forma monótona ni con
un salto en $T_0$. Un salto en $T_0$ que no acompañe a la volatilidad sí sería
pérdida de generalización.

In [ ]:
tablas_score = {
    w: ventana_movil_scores(
        Y_obs, Y_hat, T0_orig, w=w, muestras=SC_draws, li=li_s, ls=ls_s,
        t_offset=N_LAGS,
        etiquetas=[f"FPC {i + 1}" for i in COMPONENT_IDX])
    for w in VENTANAS_W
}
w_ref = VENTANAS_W[len(VENTANAS_W) // 2]
tablas_score[w_ref].to_csv(PATHS["out_report"] / f"57_ventana_scores_w{w_ref}.csv",
                           index=False)

plot_ventana_movil(
    tablas_score[w_ref], T0, ["rmse", "r2_local", "crps", "cobertura"],
    columna_grupo="componente",
    title=f"Ventana móvil por score (w={w_ref})",
    save_path=str(PATHS["out_report"] / "57_ventana_scores.png"))
plt.show()

In [ ]:
tablas_fun = {
    w: ventana_movil_funcional(X_true_ev, X_pred, grilla, T0_orig, w=w,
                               li=li_f, ls=ls_f, t_offset=N_LAGS)
    for w in VENTANAS_W
}
tablas_fun[w_ref].to_csv(PATHS["out_report"] / f"58_ventana_funcional_w{w_ref}.csv",
                         index=False)

plot_ventana_movil(
    tablas_fun[w_ref], T0, ["mise", "mise_rel", "cobertura_puntual"],
    tablas_por_w=tablas_fun,
    title="Ventana móvil del error funcional (contra la curva verdadera)",
    save_path=str(PATHS["out_report"] / "58_ventana_funcional.png"))
plt.show()

In [ ]:
# El MISE local contra la volatilidad verdadera de la misma ventana. Si el
# error sigue a sigma^2, la variación del MISE a lo largo de la serie es una
# propiedad del proceso y no una degradación del modelo.
t_ref = tablas_fun[w_ref]
centro = t_ref["t_fin"].to_numpy() if "t_fin" in t_ref.columns else None
if centro is not None:
    vol_ventana = np.array([
        sigma2_med[max(0, int(tf) - w_ref):int(tf)].mean() for tf in centro])
    _c = np.corrcoef(t_ref["mise"].to_numpy(), vol_ventana)[0, 1]

    fig, ax = plt.subplots(figsize=(11, 3.4))
    ax.plot(centro, t_ref["mise"], lw=1.2, color="#2c3e50", label="MISE local")
    ax2 = ax.twinx()
    ax2.plot(centro, vol_ventana, lw=1.2, color="#c0392b", alpha=0.8,
             label=r"$\overline{\sigma^2}$ de la ventana")
    ax.axvline(T0, color="k", ls="--", lw=1)
    ax.set_xlabel("$t$ final de la ventana"); ax.set_ylabel("MISE")
    ax2.set_ylabel(r"$\overline{\sigma^2}$ verdadera", color="#c0392b")
    ax.set_title(f"MISE local vs volatilidad verdadera (w={w_ref}) — "
                 f"corr = {_c:+.3f}", fontsize=11)
    fig.tight_layout()
    fig.savefig(PATHS["out_report"] / "59_mise_vs_volatilidad.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print(f"corr(MISE local, sigma^2 de la ventana) = {_c:+.3f}")
    print("   Alta y positiva ⇒ la variación del error a lo largo de la serie "
          "es del proceso,\n   no del modelo.")
else:
    print("La tabla de ventana móvil no trae `t_fin`; se omite la figura.")

In [ ]:
# Lectura numérica del salto en T0, excluyendo las ventanas que lo cruzan.
print(f"{'w':>4}  {'MISE train':>11}  {'MISE test':>11}  {'salto':>7}")
print(f"{'─'*4}  {'─'*11}  {'─'*11}  {'─'*7}")
for w in VENTANAS_W:
    t_ = tablas_fun[w]
    limpio = t_[~t_["cruza_T0"]]
    a = limpio.loc[limpio.bloque == "train", "mise"].mean()
    b = limpio.loc[limpio.bloque == "test",  "mise"].mean()
    print(f"{w:>4}  {a:>11.6f}  {b:>11.6f}  {b/a:>6.2f}×")

_r_vol = sigma2_med[T0:].mean() / sigma2_med[:T0].mean()
print(f"\nrazón de volatilidad test/train = {_r_vol:.2f}×")
print("   Un salto de MISE del mismo orden que esta razón es atribuible al "
      "proceso;\n   un salto claramente mayor sí es pérdida de generalización.")

## 8. Calibración marginal

Bajo calibración perfecta el PIT es uniforme. La **forma** dice qué falla: una
U indica bandas demasiado angostas, una campana demasiado anchas, y una
pendiente un sesgo del centro.

Advertencia central de este escenario: **un PIT uniforme no basta**. Un modelo
homocedástico con la varianza incondicional correcta produce un PIT
aproximadamente uniforme *marginalmente* y falla por completo condicionalmente.
La sección 9 es la que decide.

In [ ]:
pit_tr = {f"FPC {COMPONENT_IDX[k]+1}": pit_muestral(Y_obs[es_train, k],
                                                    SC_draws[:, es_train, k])
          for k in range(n_components)}
pit_te = {f"FPC {COMPONENT_IDX[k]+1}": pit_muestral(Y_obs[~es_train, k],
                                                    SC_draws[:, ~es_train, k])
          for k in range(n_components)}

plot_calibracion_pit(pit_tr, pit_te,
                     save_path=str(PATHS["out_report"] / "60_pit.png"))
plt.show()

for nombre in pit_tr:
    d_tr, d_te = diagnostico_pit(pit_tr[nombre]), diagnostico_pit(pit_te[nombre])
    print(f"  {nombre}:  train KS={d_tr['ks']:.3f} ({d_tr['forma']})   "
          f"test KS={d_te['ks']:.3f} ({d_te['forma']})")

## 9. Calibración condicional al estado verdadero

**El resultado central de esta corrida.** Eje 2 del diseño
(`docs/03 Modelo.tex §03_06`): la cobertura se estratifica según el nivel de
volatilidad **verdadero** $\sigma_t^2$ del generador —una cantidad latente que
sólo el simulador conoce y que no interviene en ninguna predicción—, en tres
estratos por cuantiles.

Qué esperar de cada resultado:

- **Cobertura ≈ nominal en los tres estratos** con anchos que crecen del
  estrato bajo al alto: el modelo captura la heterocedasticidad. Es el
  resultado que el escenario existe para producir.
- **Cobertura alta en «baja» y baja en «alta»**, con anchos parecidos entre
  estratos: la predictiva es de ancho esencialmente constante y compensa; la
  marginal lo esconde. Es el modo de fallo característico de un método lineal
  homocedástico, y verlo aquí en el PSBPM-FD indicaría que la mezcla no está
  modulando la escala.
- **Cobertura alta en los tres con anchos grandes**: calibración por exceso de
  ancho; el CRPS y el puntaje de energía por estrato lo penalizan y por eso se
  reportan junto a la cobertura.

Los cortes de cuantil se calculan sobre **todos** los orígenes evaluados, de
modo que las etiquetas «baja/media/alta» signifiquen lo mismo en ambos bloques.

In [ ]:
N_ESTRATOS = int(ESTRAT_CFG.get("n_estratos", 3))
ETIQUETAS  = list(ESTRAT_CFG.get("etiquetas", ["baja", "media", "alta"]))

estratos, ETIQUETAS = estratos_por_cuantil(sigma2_ev, N_ESTRATOS, ETIQUETAS)

resumen_estratos = pd.DataFrame({
    "estrato": ETIQUETAS,
    "n_total": [int((estratos == i).sum()) for i in range(N_ESTRATOS)],
    "n_train": [int(((estratos == i) & es_train).sum()) for i in range(N_ESTRATOS)],
    "n_test":  [int(((estratos == i) & ~es_train).sum()) for i in range(N_ESTRATOS)],
    "sigma2_medio": [float(sigma2_ev[estratos == i].mean()) for i in range(N_ESTRATOS)],
})
display(resumen_estratos.style.format({"sigma2_medio": "{:.4f}"})
        .set_caption("Estratos por volatilidad verdadera "
                     "(cortes sobre todos los orígenes evaluados)"))

if resumen_estratos["n_test"].min() < 10:
    print("⚠ Algún estrato tiene menos de 10 orígenes de prueba: su cobertura "
          "tiene\n  error de muestreo grande. Bajar N_ESTRATOS o leerla junto "
          "al bloque completo.")

In [ ]:
# ── Cobertura condicional de la banda FUNCIONAL, por bloque ──────────────────
filas_cc = []
for mask, etq_bloque in ((es_train, "train"), (~es_train, "test"),
                         (np.ones(n_orig, bool), "todo")):
    fs = cobertura_condicional(
        X_true_ev[mask], li_f[mask], ls_f[mask], estratos[mask],
        etiquetas=[ETIQUETAS[i] for i in np.unique(estratos[mask])],
        tau=grilla, muestras=X_draws[:, mask, :])
    for f in fs:
        filas_cc.append({"bloque": etq_bloque, **f})

cc_df = pd.DataFrame(filas_cc).set_index(["bloque", "estrato"])
cc_df.to_csv(PATHS["out_report"] / "61_cobertura_condicional_funcional.csv")

display(cc_df.style
    .format({"cobertura": "{:.4f}", "ancho_medio": "{:.4f}",
             "desvio": "{:+.4f}", "energy": "{:.4f}"})
    .background_gradient(subset=["cobertura"], cmap="RdYlGn", vmin=0.80, vmax=1.0)
    .set_caption(f"Cobertura funcional al {NIVEL:.0%} por estrato de volatilidad "
                 "VERDADERA · `desvio` = estrato − marginal"))

_te = cc_df.loc["test"]
print(f"cobertura marginal (test)   : "
      f"{fun_df.loc['test', f'Cob{int(NIVEL*100)}_puntual']:.4f}")
print(f"rango entre estratos (test) : "
      f"[{_te['cobertura'].min():.4f}, {_te['cobertura'].max():.4f}]   "
      f"amplitud {_te['cobertura'].max() - _te['cobertura'].min():.4f}")
print(f"razón de anchos alto/bajo   : "
      f"{_te['ancho_medio'].iloc[-1] / _te['ancho_medio'].iloc[0]:.2f}×   "
      f"(razón de sigma verdadera "
      f"{np.sqrt(resumen_estratos['sigma2_medio'].iloc[-1] / resumen_estratos['sigma2_medio'].iloc[0]):.2f}×)")

_amp = _te["cobertura"].max() - _te["cobertura"].min()
print("\nVEREDICTO del eje 2: "
      + ("✓ cobertura estable entre estratos" if _amp < 0.05 else
         "⚠ la cobertura depende del estrato: la marginal está compensando"))

In [ ]:
# ── Cobertura condicional por SCORE, bloque de prueba ────────────────────────
filas_cs = []
for k in range(n_components):
    m = ~es_train
    fs = cobertura_condicional(
        Y_obs[m, k], li_s[m, k], ls_s[m, k], estratos[m],
        etiquetas=[ETIQUETAS[i] for i in np.unique(estratos[m])],
        muestras=SC_draws[:, m, k])
    for f in fs:
        filas_cs.append({"FPC": f"FPC {COMPONENT_IDX[k] + 1}", **f})

cs_df = pd.DataFrame(filas_cs).set_index(["FPC", "estrato"])
cs_df.to_csv(PATHS["out_report"] / "62_cobertura_condicional_scores.csv")

display(cs_df.style
    .format({"cobertura": "{:.4f}", "ancho_medio": "{:.4f}",
             "desvio": "{:+.4f}", "crps": "{:.4f}"})
    .background_gradient(subset=["cobertura"], cmap="RdYlGn", vmin=0.80, vmax=1.0)
    .set_caption("Cobertura por score y estrato de volatilidad · bloque de prueba"))

In [ ]:
# ── Figura del eje 2 ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

# (a) ancho de la banda funcional vs sigma verdadera, origen a origen
ancho_f = (ls_f - li_f).mean(axis=1)
sig_ev  = np.sqrt(sigma2_ev)
ax = axes[0]
ax.scatter(sig_ev[es_train], ancho_f[es_train], s=12, alpha=0.55,
           color="#2c3e50", label="train")
ax.scatter(sig_ev[~es_train], ancho_f[~es_train], s=12, alpha=0.75,
           color="#c0392b", label="test")
_c = np.corrcoef(sig_ev, ancho_f)[0, 1]
ax.set_xlabel(r"$\sigma_t$ verdadera"); ax.set_ylabel("ancho medio de la banda")
ax.set_title(f"¿La banda sigue a la volatilidad?\ncorr = {_c:+.3f}", fontsize=10)
ax.legend(fontsize=8)

# (b) cobertura por estrato, train vs test
ax = axes[1]
x = np.arange(N_ESTRATOS); ancho_barra = 0.38
for despl, bloque, color in ((-ancho_barra/2, "train", "#2c3e50"),
                             (ancho_barra/2, "test", "#c0392b")):
    vals = [cc_df.loc[(bloque, e), "cobertura"] for e in ETIQUETAS]
    ax.bar(x + despl, vals, ancho_barra, label=bloque, color=color, alpha=0.85)
ax.axhline(NIVEL, color="k", ls="--", lw=1.2)
ax.text(x[-1], NIVEL, f" nominal {NIVEL:.0%}", fontsize=8, va="bottom", ha="right")
ax.set_xticks(x); ax.set_xticklabels(ETIQUETAS)
ax.set_xlabel("estrato de volatilidad verdadera"); ax.set_ylabel("cobertura")
ax.set_ylim(0.6, 1.02); ax.legend(fontsize=8)
ax.set_title("Cobertura condicional al estado", fontsize=10)

# (c) ancho medio por estrato contra sigma verdadera del estrato
ax = axes[2]
anchos = [cc_df.loc[("test", e), "ancho_medio"] for e in ETIQUETAS]
sigmas = np.sqrt(resumen_estratos["sigma2_medio"].to_numpy())
ax.bar(x, anchos, 0.6, color="#8e44ad", alpha=0.85)
ax2 = ax.twinx()
ax2.plot(x, sigmas, "o--", color="#16a085", lw=1.5)
ax2.set_ylabel(r"$\sigma$ verdadera del estrato", color="#16a085")
ax.set_xticks(x); ax.set_xticklabels(ETIQUETAS)
ax.set_ylabel("ancho medio (test)")
ax.set_title("Ancho de banda vs volatilidad del estrato", fontsize=10)

fig.suptitle("Eje 2 — calibración condicional al estado verdadero del generador",
             fontsize=12)
fig.tight_layout()
fig.savefig(PATHS["out_report"] / "63_calibracion_condicional.png",
            dpi=150, bbox_inches="tight")
plt.show()

## 10. Comparación con las líneas base

In [ ]:
_bl = PATHS["out_report"] / "30_baselines_test.csv"
rmse_psbp = met_df.loc["test", "RMSE"].mean()
print(f"RMSE promedio en scores — PSBP-FD (test): {rmse_psbp:.4f}\n")

if _bl.exists():
    baselines_df = pd.read_csv(_bl, index_col=0)
    display(baselines_df.style.format("{:.4f}", na_rep="—")
            .set_caption("Líneas base sobre el bloque de prueba (h=1)"))
    if "RMSE_scores_prom" in baselines_df.columns:
        for modelo, fila in baselines_df.iterrows():
            marca = "✓ PSBP mejor" if rmse_psbp < fila["RMSE_scores_prom"] else "✗ baseline mejor"
            print(f"  vs {modelo:24s} RMSE={fila['RMSE_scores_prom']:.4f}  → {marca}")
else:
    print("⚠ falta 30_baselines_test.csv — ejecuta 12_01 §6.")

print("\nLectura para el Escenario 2: la media incondicional es aquí la "
      "predicción puntual\nÓPTIMA, de modo que empatarle en RMSE es el "
      "resultado esperado y no un mal resultado.\nLa ventaja del PSBPM-FD, si "
      "existe, está en §4 (CRPS, energía) y sobre todo en §9,\ncantidades que "
      "las líneas base actuales no producen porque no entregan una\n"
      "predictiva de ancho variable.")
print("\nNota: sólo hay dos líneas base (media incondicional y persistencia). "
      "FAR(1),\nVAR sobre scores y ARIMA por score siguen pendientes en "
      "fit/baselines.py.")

In [ ]:
# ── Resumen ejecutable del experimento ───────────────────────────────────────
_te = cc_df.loc["test"]
resumen = {
    "experiment_id": EXPERIMENT_ID,
    "escenario_id": int(ESCENARIO_ID),
    "objetivo_evaluacion": OBJETIVO,
    "modo_residuo": MODO_RESIDUO,
    "nivel": NIVEL,
    "S_scores": int(S_total), "S_funcional": int(X_draws.shape[0]),
    "rmse_scores_train": float(met_df.loc["train", "RMSE"].mean()),
    "rmse_scores_test":  float(met_df.loc["test", "RMSE"].mean()),
    "r2_scores_test":    float(met_df.loc["test", "R2"].mean()),
    "crps_scores_test":  float(met_df.loc["test", "CRPS"].mean()),
    "mise_train": float(fun_df.loc["train", "MISE"]),
    "mise_test":  float(fun_df.loc["test", "MISE"]),
    "mise_truncamiento_test": float(fun_df.loc["test", "MISE_truncamiento"]),
    "energy_test": float(fun_df.loc["test", "energy"]),
    "cobertura_puntual_test": float(fun_df.loc["test", f"Cob{int(NIVEL*100)}_puntual"]),
    # Eje 2: lo que distingue a este escenario
    "cobertura_estrato_bajo":  float(_te["cobertura"].iloc[0]),
    "cobertura_estrato_alto":  float(_te["cobertura"].iloc[-1]),
    "amplitud_cobertura_estratos": float(_te["cobertura"].max() - _te["cobertura"].min()),
    "razon_ancho_alto_bajo": float(_te["ancho_medio"].iloc[-1] / _te["ancho_medio"].iloc[0]),
    "corr_ancho_sigma": float(np.corrcoef(np.sqrt(sigma2_ev),
                                          (ls_f - li_f).mean(axis=1))[0, 1]),
}
pd.Series(resumen).to_csv(PATHS["out_report"] / "64_resumen.csv", header=False)
for k, v in resumen.items():
    print(f"  {k:30s}: {v}")
print(f"\nFiguras y tablas en {PATHS['out_report']}")